# MedPredict AI — 07. Demand Analysis & Time-Series Forecasting

## Objectives
- Train machine learning models (LightGBM, XGBoost, Random Forest, Ridge) on chronological train sets.
- Generate multi-step point forecasts for pharmaceutical sales categories.
- Evaluate with MAE, RMSE, MAPE, and R².

In [3]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
from src.models.model_utils import time_series_train_test_split
from src.models.demand_forecasting import DemandForecaster
from src.evaluation.forecasting_metrics import evaluate_forecast, print_forecast_metrics
from src.visualization.model_visualizations import plot_actual_vs_predicted

ModuleNotFoundError: No module named 'pandas'

### 1. Load Processed Feature Matrix

In [ ]:
feat_path = project_root / "data" / "processed" / "forecasting_features.csv"
if feat_path.exists():
    df = pd.read_csv(feat_path)
    train_df, test_df = time_series_train_test_split(df, test_size=0.2, date_col='date')
    
    # Select target and feature columns
    target_col = 'M01AB' if 'M01AB' in df.columns else df.select_dtypes(include=['number']).columns[0]
    drop_cols = ['date', target_col]
    feature_cols = [c for c in df.select_dtypes(include=['number']).columns if c not in drop_cols]
    
    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]
    
    # Train LightGBM Forecaster
    forecaster = DemandForecaster(model_type="lightgbm")
    forecaster.fit(X_train, y_train, target_name=target_col)
    preds = forecaster.predict(X_test)
    
    # Evaluate
    metrics = evaluate_forecast(y_test, preds)
    print_forecast_metrics(metrics, model_name="LightGBM Regressor")
    
    # Visualize
    plot_actual_vs_predicted(y_test.values, preds, title=f"Forecast vs Actual for {target_col}")
    plt.show()
else:
    print(f"Please run notebook 05 to generate {feat_path}")